In [2]:
import chipwhisperer as cw

def calculate_glitch_timing(scope):
    src = scope.glitch.clk_src
    if src == "clkgen":
        f_glitch = scope.clock.clkgen_freq
    elif src == "target":
        f_glitch = scope.clock.freq_ctr        # measured target clock
    elif src == "pll":
        f_glitch = scope.clock.adc_freq        # verify by measurement
                                               # Verify it by measuring with your LA (set ext_offset = 100, measure the trigger-to-glitch delay, divide by 100).
    else:
        raise ValueError(f"unexpected glitch clk_src: {src!r}")

    f_glitch = float(f_glitch)
    if f_glitch <= 0:
        raise RuntimeError(f"glitch clock frequency is {f_glitch}; is the clock present?")
    T = 1 / f_glitch

    div = scope.glitch.phase_shift_steps

    t_ext_offset = scope.glitch.ext_offset * T
    t_offset     = (scope.glitch.offset / div) * T
    t_width      = (scope.glitch.width  / div) * T
    t_repeat     = scope.glitch.repeat * T
    t_total      = t_ext_offset + t_offset

    print(f"glitch starts ~{t_total*1e9:.2f} ns after trigger")
    print(f"ext_offset {t_ext_offset*1e9:.2f} ns offset {t_offset*1e9:.2f} ns width {t_width*1e9:.2f} ns") 
    
    return {
        "T": T,
        "ext_offset": t_ext_offset,
        "offset": t_offset,
        "width": t_width,
        "repeat": t_repeat,
        "total": t_total,
    }


In [5]:
#import chipwhisperer as cw
#scope = cw.scope()
#help(scope.glitch)

Help on GlitchSettings in module chipwhisperer.capture.scopes.cwhardware.ChipWhispererGlitch object:

class GlitchSettings(chipwhisperer.common.utils.util.DisableNewAttr)
 |  GlitchSettings(cwglitch)
 |
 |  Method resolution order:
 |      GlitchSettings
 |      chipwhisperer.common.utils.util.DisableNewAttr
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, cwglitch)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(self)
 |      Return repr(self).
 |
 |  __str__(self)
 |      Return str(self).
 |
 |  manualTrigger(self)
 |
 |  manual_trigger(self) -> None
 |      Manually trigger the glitch output.
 |
 |      This method only inserts a glitch in manual trigger mode. In this
 |      mode, it is likewise the only way to insert a glitch.
 |
 |  readStatus(self) -> tuple
 |      Read the status of the two glitch DCMs.
 |
 |      Returns:
 |          A tuple with 4 elements::
 |
 |           * phase1: Phase shift of DCM1 (N/A fo